# Project 3 — Text Vectorization & Classification

## Why vectorize?
Machine-learning models cannot read words — they only understand **numbers**.
We must convert each document into a numeric vector first.

## Two classic vectorization methods

### 1. Bag of Words (BoW)
Build a vocabulary of every unique word, then count how many times each word appears in each document.

| Doc | love | movie | hate | bad |
|---|---|---|---|---|
| *I love this movie* | 1 | 1 | 0 | 0 |
| *I hate this movie* | 0 | 1 | 1 | 0 |

### 2. TF-IDF
Term-Frequency × Inverse-Document-Frequency.
Words that are **frequent in one document** but **rare across the corpus** get a *high* score (they are discriminative).
Words like *the*, *is*, *and* get *low* scores.

$$\text{tfidf}(t,d) = \text{tf}(t,d) \times \log\frac{N}{\text{df}(t)}$$

## Our task
Classify a sentence as **Sports (0)** or **Technology (1)**.

## Step 1 — Imports

In [1]:
import nltk, spacy, re, string
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

for pkg in ['stopwords', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm')

## Step 2 — Tiny labelled dataset

In [2]:
texts = [
    # Sports (label = 0)
    'The football match between Real Madrid and Barcelona was thrilling',
    'Cricket world cup final is being played in India this Sunday',
    'The basketball player scored 40 points in the NBA finals',
    'He won the gold medal in the Olympic swimming championship',
    'Tennis legend Roger Federer announced his retirement from the sport',
    # Tech (label = 1)
    'Apple released a new iPhone with an advanced AI chip and better camera',
    'Google launched a new large language model competing with ChatGPT',
    'Microsoft is investing billions of dollars in cloud computing and AI',
    'The latest Android update brings better security and battery life',
    'Tesla unveiled a new self-driving car powered by neural networks',
]
labels = [0]*5 + [1]*5
label_names = {0: 'Sports', 1: 'Technology'}

## Step 3 — Clean the text

We define cleaners using both NLTK and spaCy, and use the spaCy version (because it lemmatizes).

In [3]:
def clean_with_nltk(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    stop = set(stopwords.words('english'))
    return ' '.join(t for t in tokens if t not in stop and len(t) > 1)

def clean_with_spacy(text):
    doc = nlp(text.lower())
    return ' '.join(tok.lemma_ for tok in doc
                    if not tok.is_stop and not tok.is_punct and not tok.is_space)

print('Original :', texts[0])
print('NLTK     :', clean_with_nltk(texts[0]))
print('spaCy    :', clean_with_spacy(texts[0]))

clean_texts = [clean_with_nltk(t) for t in texts]

Original : The football match between Real Madrid and Barcelona was thrilling
NLTK     : football match real madrid barcelona thrilling
spaCy    : football match real madrid barcelona thrill


## Step 4 — Vectorize with Bag of Words

In [4]:
bow = CountVectorizer()
X_bow = bow.fit_transform(clean_texts)

print('Vocabulary size:', len(bow.vocabulary_))
print('Matrix shape   :', X_bow.shape)
print('First 15 words :', bow.get_feature_names_out()[:15])
print('Doc 0 vector   :', X_bow.toarray()[0][:15])

Vocabulary size: 68
Matrix shape   : (10, 68)
First 15 words : ['40' 'advanced' 'ai' 'android' 'announced' 'apple' 'barcelona'
 'basketball' 'battery' 'better' 'billions' 'brings' 'camera' 'car'
 'championship']
Doc 0 vector   : [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]


## Step 5 — Vectorize with TF-IDF

In [ ]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(clean_texts)

row = X_tfidf.toarray()[0]
vocab = tfidf.get_feature_names_out()
print('Document 0 non-zero TF-IDF values:')
for i, v in enumerate(row):
    if v > 0:
        print(f'  {vocab[i]:<15} = {v:.3f}')

## Step 6 — Train classifiers

**Multinomial Naive Bayes** and **Logistic Regression** are both strong baselines for text.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, labels, test_size=0.2, random_state=42, stratify=labels)

nb = MultinomialNB().fit(X_train, y_train)
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)

print(f'Naive Bayes  acc = {accuracy_score(y_test, nb.predict(X_test)):.2f}')
print(f'Logistic Reg acc = {accuracy_score(y_test, lr.predict(X_test)):.2f}')

## Step 7 — Predict new sentences

**IMPORTANT:** apply the *same* cleaner and use `transform()` (not `fit_transform()`) on new data.

In [ ]:
new_sentences = [
    "Lionel Messi scored a hat-trick in last night's match",
    "Nvidia's new GPU is powering generative AI breakthroughs",
    'The Indian cricket team won the tournament after a tight final',
    'OpenAI released a new version of ChatGPT with improved reasoning',
]
cleaned_new = [clean_with_spacy(s) for s in new_sentences]
X_new = tfidf.transform(cleaned_new)
predictions = lr.predict(X_new)

for s, p in zip(new_sentences, predictions):
    print(f'  "{s[:55]}…" → {label_names[p]}')

## Summary

Pipeline:
1. Collect labelled text
2. Clean it
3. Vectorize (BoW or TF-IDF)
4. Train (Naive Bayes / Logistic Regression)
5. Predict on new data

**Golden rule:** `fit_transform()` only on training data, `transform()` on test/new data.